# Aquaculture Model Analysis

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Import our custom modules
import sys
sys.path.append('src')

from aquaculture.feature_engineering import AquacultureFeatureEngineer
from src.inference import load_inference_pipeline
from src.plotting import (
    plot_feature_importance, plot_roc_curve, plot_precision_recall_curve,
    plot_confusion_matrix, plot_calibration_curve
)
from src.metrics import calculate_metrics, competition_score

# For reproducibility
import random
np.random.seed(42)
random.seed(42)

# Set up paths
DATA_DIR = Path('../data')
EXPERIMENTS_DIR = Path('../experiments')

# Try to find the most recent experiment directory
if EXPERIMENTS_DIR.exists():
    experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
    if experiment_dirs:
        # Sort by modification time (newest first)
        experiment_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_experiment = experiment_dirs[0]
        print(f"Found experiment: {latest_experiment.name}")
    else:
        print("No experiment directories found!")
        return
else:
    print("Experiments directory not found!")
    return

# Load the inference pipeline
print("Loading inference pipeline...")
try:
    pipeline = load_inference_pipeline(str(latest_experiment))
    print("✓ Inference pipeline loaded successfully")
    print(f"Model type: {type(pipeline.model).__name__}")
    if pipeline.feature_names:
        print(f"Number of features: {len(pipeline.feature_names)}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check that the experiment directory exists and contains a trained model")
    return


## 2. Load Training Data for Analysis

Load training data to evaluate model performance and perform analysis.

In [ ]:
# Load training data
print("Loading training data...")
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
print(f"Training data shape: {train_df.shape}")

# Load features if they exist, otherwise create them
FEATURES_FILE = EXPERIMENTS_DIR / latest_experiment.name / 'features.parquet'
if FEATURES_FILE.exists():
    print("Loading pre-computed features...")
    features_df = pd.read_parquet(FEATURES_FILE)
else:
    print("Computing features...")
    feature_engineer = AquacultureFeatureEngineer()
    features_df = feature_engineer.transform(train_df)
    features_df.to_parquet(FEATURES_FILE)
    print(f"Features saved to {FEATURES_FILE}")

# Prepare data for analysis
print("Preparing data for analysis...")
# Remove target columns from features
feature_cols = [col for col in features_df.columns if not col.startswith('target_')]
X = features_df[feature_cols].values

# Get target variables
y = {}
for i in range(1, 4):
    target_col = f'target_{i}'
    if target_col in train_df.columns:
        y[i] = train_df[target_col].values

# Make predictions
print("Making predictions...")
predictions = pipeline.predict(X)
probabilities = pipeline.predict_proba(X)


## 3. Evaluate Model Performance

Calculate various metrics to evaluate the model's performance.

In [ ]:
# Calculate metrics for each target
metrics_dict = {}
for i in range(1, 4):
    print(f"\n=== Target {i} ===")
    target_pred = predictions[i]
    target_prob = probabilities[i][:, 1]  # Probability of positive class
    target_true = y[i]
    
    # Calculate various metrics
    metrics = calculate_metrics(target_true, target_pred, target_prob)
    metrics_dict[i] = metrics
    
    # Print key metrics
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1']:.4f}")
    print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
    print(f"PR AUC:    {metrics['pr_auc']:.4f}")

# Calculate competition score
competition_score = competition_score(y, predictions)
print(f"\nCompetition Score: {competition_score:.4f}")


## 4. Generate Visualizations

Create diagnostic plots to understand model performance.

In [ ]:
# Generate plots for each target
for i in range(1, 4):
    print(f"\nGenerating plots for target {i}...")
    target_pred = predictions[i]
    target_prob = probabilities[i][:, 1]
    target_true = y[i]
    
    # Create a directory for plots
    PLOTS_DIR = EXPERIMENTS_DIR / latest_experiment.name / 'plots'
    PLOTS_DIR.mkdir(exist_ok=True)
    
    # ROC Curve
    plot_roc_curve(target_true, target_prob, 
                   title=f'ROC Curve - Target {i}',
                   save_path=PLOTS_DIR / f'roc_curve_target_{i}.png')
    
    # Precision-Recall Curve
    plot_precision_recall_curve(target_true, target_prob,
                                title=f'Precision-Recall Curve - Target {i}',
                                save_path=PLOTS_DIR / f'pr_curve_target_{i}.png')
    
    # Confusion Matrix
    plot_confusion_matrix(target_true, target_pred,
                          title=f'Confusion Matrix - Target {i}',
                          save_path=PLOTS_DIR / f'confusion_matrix_target_{i}.png')
    
    # Calibration Curve
    plot_calibration_curve(target_true, target_prob,
                           title=f'Calibration Curve - Target {i}',
                           save_path=PLOTS_DIR / f'calibration_curve_target_{i}.png')

print("\nAll plots generated successfully!")
